# 04 — LLM Client (OpenRouter)

Tests `infrastructure/llm/client.py`:
- Singleton client construction
- Non-streaming `chat()` call
- Streaming `chat_stream()` call
- JSON mode response

> **Requires**: `OPENROUTER_API_KEY` in environment (or `.env` at project root).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'))

api_key = os.getenv('OPENROUTER_API_KEY')
assert api_key, 'OPENROUTER_API_KEY not set — add it to .env at the project root'
print('API key found:', api_key[:8] + '...')

In [ ]:
from infrastructure.llm.client import chat, chat_stream, _get_client
from utils.config import CLAUDE_MODEL, CLAUDE_MAX_TOKENS_CLASSIFY

## 1. Singleton client

In [ ]:
client1 = _get_client()
client2 = _get_client()
assert client1 is client2, 'Client should be a singleton'
print('Singleton test: PASSED')

## 2. Non-streaming chat call

In [ ]:
response = chat(
    system='You are a helpful assistant. Reply very briefly.',
    messages=[{'role': 'user', 'content': 'Say "hello" and nothing else.'}],
    max_tokens=20,
    model=CLAUDE_MODEL,
)

print('Response:', repr(response))
assert isinstance(response, str), 'chat() must return a string'
assert len(response) > 0, 'Response should not be empty'
print('Non-streaming chat: PASSED')

## 3. JSON mode

In [ ]:
import json

response = chat(
    system='Return JSON only.',
    messages=[{'role': 'user', 'content': 'Return {"status": "ok"}'}],
    max_tokens=50,
    model=CLAUDE_MODEL,
    json_mode=True,
)

print('Raw JSON response:', repr(response))
parsed = json.loads(response)
print('Parsed:', parsed)
assert isinstance(parsed, dict), 'JSON mode should return parseable JSON'
print('JSON mode: PASSED')

## 4. Streaming chat call

In [ ]:
chunks = []
for chunk in chat_stream(
    system='You are a helpful assistant.',
    messages=[{'role': 'user', 'content': 'Count from 1 to 5.'}],
    max_tokens=60,
    model=CLAUDE_MODEL,
):
    chunks.append(chunk)
    print(chunk, end='', flush=True)

print()  # newline after stream
full_response = ''.join(chunks)

assert len(chunks) > 1, 'Streaming should produce multiple chunks'
assert len(full_response) > 0, 'Full response must not be empty'
print(f'\nStreaming chat: PASSED ({len(chunks)} chunks, {len(full_response)} chars)')

## 5. Conversation history is forwarded correctly

In [ ]:
# Multi-turn: the model should remember what was said earlier in the messages list
response = chat(
    system='You are a helpful assistant.',
    messages=[
        {'role': 'user',      'content': 'My secret number is 42.'},
        {'role': 'assistant', 'content': 'Got it, your secret number is 42.'},
        {'role': 'user',      'content': 'What is my secret number?'},
    ],
    max_tokens=30,
    model=CLAUDE_MODEL,
)

print('Response:', response)
assert '42' in response, 'Model should recall the secret number from history'
print('Multi-turn history: PASSED')